## Qwen2.5-3B-Instruct

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ========================== CELL 1: INSTALL & LOGIN ==========================
# CELL 1: Install & Login
!pip install -q transformers accelerate bitsandbytes torch pandas tqdm

from huggingface_hub import login
login()  # Paste your HF token (read access to gated models if needed — Qwen2.5-3B is public)

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import json
import pandas as pd
from tqdm import tqdm
from google.colab import files



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.4 MB/s eta 0:00:00


In [ ]:
# ========================== CELL 2: LOAD Qwen2.5-3B-INSTRUCT (4-bit) ==========================
# CELL 2: Load Qwen2.5-3B-Instruct (4-bit)
model_name = "Qwen/Qwen2.5-3B-Instruct"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=quant_config,
    torch_dtype=torch.bfloat16,
)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
# ========================== CELL 3: DEFINE 3 LLM AGENTS ==========================
# CELL 3: Define 3 LLM Agents
def query_agent(system_prompt, user_message, max_new=180):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_message}
    ]

    # Qwen2.5 chat template
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new,
            do_sample=False,           # greedy → stable JSON
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    ).strip()

    return response

# ─── Agent 1: Fact Checker ─────────────────────────────────────
fact_system = """你是一位事實檢查代理 (FactCheckerAgent)。
根據評論文字、標題與 URL (若有) 判斷這則評論：
- real → 真實、與產品/服務相關的正常評論
- fake(misinformation) → 明顯錯誤、造假、誤導或廣告垃圾
- unrelated → 完全離題、無關產品

若 URL 與 title 為空或 null，僅依文字內容判斷是否像正常評論。
僅輸出以下 JSON，勿多說任何文字：
{"label": "real"} 或 {"label": "fake(misinformation)"} 或 {"label": "unrelated"}"""

# ─── Agent 2: Sentiment Analyzer ───────────────────────────────
sentiment_system = """你是一位情緒分析代理 (SentimentAgent)。
分析這則評論的情緒（支援英文與繁體中文、表情符號、連結）：
僅輸出以下 JSON，勿多說任何文字：
{"sentiment": "positive"} 或 {"sentiment": "negative"} 或 {"sentiment": "neutral"}"""

# ─── Agent 3: Combiner (final decision) ────────────────────────
combiner_system = """你是最終整合代理 (CombinerAgent)。
收到：
- 標題、URL、文字
- Fact label
- Sentiment

除非明顯錯誤，否則保留原值；若有問題則修正。
僅輸出以下 JSON，勿多說任何文字：
{"label": "...", "sentiment": "..."}"""


In [ ]:
# ========================== CELL 4: LOAD DATA & RUN 3 AGENTS ==========================
# CELL 4: Load data (only first 100 for testing)
# Upload your data.jsonl to /content/data.jsonl first
data_path = "/content/drive/MyDrive/Czech/M12_final_v1_to_dataset.jsonl"

reviews = []
with open(data_path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if line.strip():
            try:
                reviews.append(json.loads(line))
            except:
                continue
        if len(reviews) >= 100:
            break

print(f"Loaded {len(reviews)} reviews (limited to first 100 for testing)")

results = []

for row in tqdm(reviews, desc="Processing with 3 Qwen2.5 Agents"):
    package_id = row.get("package_id", "")
    review_id  = row.get("review_id",  "")
    cid        = row.get("cid",        "")
    text       = row.get("text",       "")
    url        = row.get("url")   if row.get("url")   is not None else ""
    title      = row.get("title") if row.get("title") is not None else ""

    if not text.strip():
        label = "unrelated"
        sentiment = "neutral"
    else:
        # Agent 1 ─ Fact check
        fact_user = f"標題: {title}\n網址: {url}\n評論: {text}"
        fact_raw = query_agent(fact_system, fact_user)
        try:
            fact_json = json.loads(fact_raw)
            label = fact_json.get("label", "unrelated")
        except:
            label = "unrelated"

        # Agent 2 ─ Sentiment
        sent_user = f"評論: {text}"
        sent_raw = query_agent(sentiment_system, sent_user)
        try:
            sent_json = json.loads(sent_raw)
            sentiment = sent_json.get("sentiment", "neutral")
        except:
            sentiment = "neutral"

        # Agent 3 ─ Combiner / final check
        combiner_user = f"""標題: {title}
網址: {url}
評論: {text}
事實驗證標籤: {label}
情緒: {sentiment}
請給出最終 JSON。"""
        final_raw = query_agent(combiner_system, combiner_user, max_new=120)
        try:
            final_json = json.loads(final_raw)
            label      = final_json.get("label", label)
            sentiment  = final_json.get("sentiment", sentiment)
        except:
            pass

    results.append({
        "package_id": package_id,
        "review_id":  review_id,
        "cid":        cid,
        "url":        url,
        "title":      title,
        "text":       text,
        "label":      label,
        "sentiment":  sentiment
    })



Loaded 100 reviews (limited to first 100 for testing)


Processing with 3 Qwen2.5 Agents: 100%|██████████| 100/100 [07:17<00:00,  4.37s/it]


In [ ]:
# ========================== CELL 5: SAVE CSV ==========================

# CELL 5: Save & Download CSV
df = pd.DataFrame(results)
output_path = "/content/drive/MyDrive/Czech/qwen2.5-analyzed_reviews.csv"
df.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"Done! Saved {len(results)} rows → {output_path}")
print(df.head(8))

# Optional: download
from google.colab import files
files.download(output_path)





DONE! CSV saved to /content/drive/MyDrive/Czech/Llama-3.2-analyzed_reviews.csv
  package_id review_id        cid url title  \
0         01  ptt_30_1  ptt_30_c1             
1         01  ptt_30_2  ptt_30_c2             
2         01  ptt_30_3  ptt_30_c3             
3         01  ptt_30_4  ptt_30_c4             
4         01  ptt_30_5  ptt_30_c5             

                                    text      label sentiment  
0               都說了，就別執著了。而且也沒意義，吃不了那麼高。       real   neutral  
1          走PD3.2，可能只有17PRO能吃滿也說不定，當專用即可  unrelated   neutral  
2                 #1emPiow4 (MobileComm)   1emPiow4   neutral  
3                              typec懂的都懂  unrelated   neutral  
4  我上個月買Google Pixel Flex 67W雙C充電頭 支援AVS       real   neutral  


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done! Saved 100 rows → /content/drive/MyDrive/Czech/qwen2.5-analyzed_reviews.csv
  package_id review_id        cid url title  \
0         01  ptt_30_1  ptt_30_c1             
1         01  ptt_30_2  ptt_30_c2             
2         01  ptt_30_3  ptt_30_c3             
3         01  ptt_30_4  ptt_30_c4             
4         01  ptt_30_5  ptt_30_c5             
5         01  ptt_30_6  ptt_30_c6             
6         01  ptt_30_7  ptt_30_c7             
7         01  ptt_30_8  ptt_30_c8             

                                    text      label sentiment  
0               都說了，就別執著了。而且也沒意義，吃不了那麼高。       real   neutral  
1          走PD3.2，可能只有17PRO能吃滿也說不定，當專用即可  unrelated   neutral  
2                 #1emPiow4 (MobileComm)   1emPiow4   neutral  
3                              typec懂的都懂  unrelated   neutral  
4  我上個月買Google Pixel Flex 67W雙C充電頭 支援AVS       real   neutral  
5    https://reurl.cc/7VL655 比Apple的實用很多       real  positive  
6   可以同時充MacBook+iPhone/iPad/AirPods ...       

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>